In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import joblib

from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

In [ ]:
# @title
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "google/gemma-2-2b"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Model loaded successfully!")
print("Number of layers:", model.config.num_hidden_layers)
print("Hidden size:", model.config.hidden_size)

config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Model loaded successfully!
Number of layers: 26
Hidden size: 2304


In [ ]:
def extract_hidden_states(
    dataset_split,
    model,
    tokenizer,
    batch_size=8
):

    model.eval()

    num_layers = model.config.num_hidden_layers + 1

    representations = {
        layer: []
        for layer in range(num_layers)
    }

    labels = []

    for start in range(
        0,
        len(dataset_split),
        batch_size
    ):

        batch = dataset_split[
            start:start + batch_size
        ]

        prompts = batch["prompt"]

        batch_labels = [
            0 if safe else 1
            for safe in batch["is_safe"]
        ]

        inputs = tokenizer(
            prompts,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():

            outputs = model(
                **inputs,
                output_hidden_states=True
            )

        attention_mask = inputs[
            "attention_mask"
        ]

        for layer in range(num_layers):

            hidden = outputs.hidden_states[layer]

            mask = (
                attention_mask
                .unsqueeze(-1)
                .expand(hidden.size())
                .float()
            )

            summed = torch.sum(
                hidden * mask,
                dim=1
            )

            counts = torch.clamp(
                mask.sum(dim=1),
                min=1e-9
            )

            pooled = summed / counts

            representations[layer].append(
                pooled
                .cpu()
                .float()
                .numpy()
            )

        labels.extend(batch_labels)

        if start % (batch_size * 10) == 0:

            print(
                f"Processed "
                f"{min(start + batch_size, len(dataset_split))}"
                f"/{len(dataset_split)}"
            )

    for layer in representations:

        representations[layer] = np.concatenate(
            representations[layer],
            axis=0
        )

    labels = np.array(labels)

    return representations, labels

In [ ]:
from datasets import load_dataset

dataset = load_dataset("PKU-Alignment/BeaverTails")

print(dataset)

In [ ]:
from datasets import load_dataset

toxic_chat = load_dataset(
    "lmsys/toxic-chat",
    "toxicchat0124"
)

print(toxic_chat)

In [ ]:
from datasets import load_dataset

rtp = load_dataset(
    "allenai/real-toxicity-prompts"
)

print(rtp)

In [ ]:
from datasets import load_dataset

jigsaw = load_dataset(
    "thesofakillers/jigsaw-toxic-comment-classification-challenge"
)

print(jigsaw)

In [ ]:
import numpy as np

jigsaw_labels = np.array(jigsaw["train"]["toxic"])

print("Total:", len(jigsaw_labels))
print("Safe:", np.sum(jigsaw_labels == 0))
print("Toxic:", np.sum(jigsaw_labels == 1))

Total: 159571
Safe: 144277
Toxic: 15294


In [ ]:
SEED = 42

rng = np.random.RandomState(SEED)

safe_idx = np.where(jigsaw_labels == 0)[0]
toxic_idx = np.where(jigsaw_labels == 1)[0]

jigsaw_safe_idx = rng.choice(
    safe_idx,
    size=400,
    replace=False
)

jigsaw_toxic_idx = rng.choice(
    toxic_idx,
    size=400,
    replace=False
)

jigsaw_pool_idx = np.concatenate([
    jigsaw_safe_idx,
    jigsaw_toxic_idx
])

rng.shuffle(jigsaw_pool_idx)

print("Pool size:", len(jigsaw_pool_idx))

Pool size: 800


In [ ]:
jigsaw_for_extraction = [
    {
        "prompt": jigsaw["train"][int(idx)]["comment_text"],
        "is_safe": bool(jigsaw["train"][int(idx)]["toxic"] == 0)
    }
    for idx in jigsaw_pool_idx
]

In [ ]:
jigsaw_representations, jigsaw_labels = extract_hidden_states(
    jigsaw_for_extraction,
    model,
    tokenizer,
    batch_size=8
)

Processed 8/800
Processed 88/800
Processed 168/800
Processed 248/800
Processed 328/800
Processed 408/800
Processed 488/800
Processed 568/800
Processed 648/800
Processed 728/800


In [ ]:
jigsaw_X = jigsaw_representations[11]
jigsaw_y = jigsaw_labels

print("Jigsaw X:", jigsaw_X.shape)
print("Jigsaw y:", jigsaw_y.shape)
print("Safe:", np.sum(jigsaw_y == 0))
print("Toxic:", np.sum(jigsaw_y == 1))

Jigsaw X: (900, 2304)
Jigsaw y: (900,)
Safe: 450
Toxic: 450


In [ ]:
jigsaw_X_800 = jigsaw_representations[11]
jigsaw_y_800 = jigsaw_labels

print(jigsaw_X_800.shape)
print(jigsaw_y_800.shape)

(800, 2304)
(800,)


In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

# Use the 30k training split for our initial experiment
source = dataset["30k_train"]

# Separate safe and harmful examples
safe = source.filter(lambda x: x["is_safe"] == True)
harmful = source.filter(lambda x: x["is_safe"] == False)

print("Safe examples:", len(safe))
print("Harmful examples:", len(harmful))

Filter:   0%|          | 0/27186 [00:00<?, ? examples/s]

Filter:   0%|          | 0/27186 [00:00<?, ? examples/s]

Safe examples: 11604
Harmful examples: 15582


In [ ]:
from collections import Counter

# Count how often each category appears as True
category_counts = Counter()

for example in source:
    categories = example["category"]

    for category, value in categories.items():
        if value:
            category_counts[category] += 1

print("Harmful category counts:")
for category, count in category_counts.most_common():
    print(f"{category}: {count}")

Harmful category counts:
violence,aiding_and_abetting,incitement: 6927
non_violent_unethical_behavior: 4816
financial_crime,property_crime,theft: 2566
hate_speech,offensive_language: 2537
discrimination,stereotype,injustice: 2346
drug_abuse,weapons,banned_substance: 1527
privacy_violation: 1397
controversial_topics,politics: 907
sexually_explicit,adult_content: 704
misinformation_regarding_ethics,laws_and_safety: 652
animal_abuse: 357
terrorism,organized_crime: 294
self_harm: 205
child_abuse: 185


In [ ]:
# Take an equal number from each class
N_PER_CLASS = 2000

safe_subset = safe.shuffle(seed=42).select(range(N_PER_CLASS))
harmful_subset = harmful.shuffle(seed=42).select(range(N_PER_CLASS))

# Combine them
balanced = Dataset.from_list(
    list(safe_subset) + list(harmful_subset)
)

# Shuffle the combined dataset
balanced = balanced.shuffle(seed=42)

print("Total examples:", len(balanced))
print("Safe:", sum(balanced["is_safe"]))
print("Harmful:", sum(not x for x in balanced["is_safe"]))

Total examples: 4000
Safe: 2000
Harmful: 2000


In [ ]:
# First: 70% train, 30% temporary
train_data, temp_data = train_test_split(
    list(range(len(balanced))),
    test_size=0.30,
    random_state=42,
    stratify=balanced["is_safe"]
)

# Then split temporary into validation/test
temp_labels = [balanced[i]["is_safe"] for i in temp_data]

val_indices, test_indices = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

train_data = balanced.select(train_data)
val_data = balanced.select(val_indices)
test_data = balanced.select(test_indices)

print("Train:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))

Train: 2800
Validation: 600
Test: 600


In [ ]:
train_repr, train_labels = extract_hidden_states(
    train_data,
    model,
    tokenizer,
    batch_size=8
)

Processed 8/2800
Processed 88/2800
Processed 168/2800
Processed 248/2800
Processed 328/2800
Processed 408/2800
Processed 488/2800
Processed 568/2800
Processed 648/2800
Processed 728/2800
Processed 808/2800
Processed 888/2800
Processed 968/2800
Processed 1048/2800
Processed 1128/2800
Processed 1208/2800
Processed 1288/2800
Processed 1368/2800
Processed 1448/2800
Processed 1528/2800
Processed 1608/2800
Processed 1688/2800
Processed 1768/2800
Processed 1848/2800
Processed 1928/2800
Processed 2008/2800
Processed 2088/2800
Processed 2168/2800
Processed 2248/2800
Processed 2328/2800
Processed 2408/2800
Processed 2488/2800
Processed 2568/2800
Processed 2648/2800
Processed 2728/2800


In [ ]:
val_repr, val_labels = extract_hidden_states(
    val_data,
    model,
    tokenizer,
    batch_size=8
)

Processed 8/600
Processed 88/600
Processed 168/600
Processed 248/600
Processed 328/600
Processed 408/600
Processed 488/600
Processed 568/600


In [ ]:
test_repr, test_labels = extract_hidden_states(
    test_data,
    model,
    tokenizer,
    batch_size=8
)

Processed 8/600
Processed 88/600
Processed 168/600
Processed 248/600
Processed 328/600
Processed 408/600
Processed 488/600
Processed 568/600


In [ ]:
import pandas as pd
from datasets import Dataset

# 1. Convert ToxicChat train to a pandas DataFrame
toxichat_train_df = toxic_chat["train"].to_pandas()

# 2. Separate classes
toxichat_toxic = toxichat_train_df[
    toxichat_train_df["toxicity"] == 1
].copy()

# Determine 40% sample size per class based on the minority class
n_samples_per_class = int(len(toxichat_toxic) * 0.40)

# 3. Sample 40% from toxic and 40% from benign
toxichat_toxic_sample = toxichat_toxic.sample(
    n=n_samples_per_class,
    random_state=42
)

toxichat_benign_sample = toxichat_train_df[
    toxichat_train_df["toxicity"] == 0
].sample(
    n=n_samples_per_class,
    random_state=42
)

# 4. Combine and Shuffle
toxichat_sampled_df = pd.concat(
    [toxichat_toxic_sample, toxichat_benign_sample],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n40% Sampled ToxicChat (Total: {len(toxichat_sampled_df)} rows):")
print(toxichat_sampled_df["toxicity"].value_counts())


40% Sampled ToxicChat (Total: 306 rows):
toxicity
0    153
1    153
Name: count, dtype: int64


In [ ]:
from datasets import Dataset
import pandas as pd

# Convert ToxicChat train to a pandas DataFrame
toxichat_train_df = toxic_chat["train"].to_pandas()

print("Original ToxicChat train:")
print(toxichat_train_df["toxicity"].value_counts())

# Separate classes
toxichat_toxic = toxichat_train_df[
    toxichat_train_df["toxicity"] == 1
].copy()

toxichat_benign = toxichat_train_df[
    toxichat_train_df["toxicity"] == 0
].sample(
    n=len(toxichat_toxic),
    random_state=42
).copy()

# Balanced ToxicChat subset
toxichat_balanced_df = pd.concat(
    [
        toxichat_toxic,
        toxichat_benign
    ],
    ignore_index=True
)

# Shuffle
toxichat_balanced_df = toxichat_balanced_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\nBalanced ToxicChat:")
print(
    toxichat_balanced_df["toxicity"].value_counts()
)

print(
    "\nShape:",
    toxichat_balanced_df.shape
)

Original ToxicChat train:
toxicity
0    4698
1     384
Name: count, dtype: int64

Balanced ToxicChat:
toxicity
0    384
1    384
Name: count, dtype: int64

Shape: (768, 7)


In [ ]:
toxichat_probe_df = toxichat_balanced_df[
    ["user_input", "toxicity"]
].rename(
    columns={
        "user_input": "prompt"
    }
)

# is_safe:
# toxicity 0 → safe = True
# toxicity 1 → safe = False

toxichat_probe_df["is_safe"] = (
    toxichat_probe_df["toxicity"] == 0
)

toxichat_probe_df = toxichat_probe_df[
    ["prompt", "is_safe"]
]

toxichat_probe = Dataset.from_pandas(
    toxichat_probe_df,
    preserve_index=False
)

print(toxichat_probe)
print(toxichat_probe[0])

Dataset({
    features: ['prompt', 'is_safe'],
    num_rows: 768
})
{'prompt': 'Can I grow claws on my fingers?', 'is_safe': True}


In [ ]:
from datasets import load_dataset
from collections import Counter

# Reload ToxicChat
toxic_chat = load_dataset(
    "lmsys/toxic-chat",
    "toxicchat0124"
)

toxichat_train = toxic_chat["train"]

print(toxichat_train)
print(Counter(toxichat_train["toxicity"]))

Dataset({
    features: ['conv_id', 'user_input', 'model_output', 'human_annotation', 'toxicity', 'jailbreaking', 'openai_moderation'],
    num_rows: 5082
})
Counter({0: 4698, 1: 384})


In [ ]:
from collections import Counter

print(Counter(toxichat_train["toxicity"]))

Counter({0: 4698, 1: 384})


In [ ]:
toxichat_train = toxic_chat["train"]

print(toxichat_train.column_names)
print(Counter(toxichat_train["toxicity"]))

['conv_id', 'user_input', 'model_output', 'human_annotation', 'toxicity', 'jailbreaking', 'openai_moderation']
Counter({0: 4698, 1: 384})


In [ ]:
import pandas as pd

# Convert ToxicChat into a DataFrame
toxichat_df = pd.DataFrame({
    "prompt": toxichat_train["user_input"],
    "toxicity": toxichat_train["toxicity"]
})

# Separate the two classes
safe = toxichat_df[
    toxichat_df["toxicity"] == 0
].copy()

toxic = toxichat_df[
    toxichat_df["toxicity"] == 1
].copy()

print("Original ToxicChat:")
print(toxichat_df["toxicity"].value_counts())

print("\nSafe examples:", len(safe))
print("Toxic examples:", len(toxic))

# Balance the dataset
n = min(len(safe), len(toxic))

safe_balanced = safe.sample(
    n=n,
    random_state=42
)

toxic_balanced = toxic.sample(
    n=n,
    random_state=42
)

# Combine
toxichat_balanced = pd.concat(
    [safe_balanced, toxic_balanced],
    ignore_index=True
)

# Shuffle
toxichat_balanced = toxichat_balanced.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\nBalanced ToxicChat:")
print(toxichat_balanced["toxicity"].value_counts())
print("Shape:", toxichat_balanced.shape)

Original ToxicChat:
toxicity
0    4698
1     384
Name: count, dtype: int64

Safe examples: 4698
Toxic examples: 384

Balanced ToxicChat:
toxicity
1    384
0    384
Name: count, dtype: int64
Shape: (768, 2)


In [ ]:
from datasets import Dataset

toxichat_data = Dataset.from_dict({
    "prompt": toxichat_balanced["prompt"].tolist(),
    "is_safe": [
        False if label == 1 else True
        for label in toxichat_balanced["toxicity"]
    ]
})

print(toxichat_data)
print(toxichat_data[0])

Dataset({
    features: ['prompt', 'is_safe'],
    num_rows: 768
})
{'prompt': 'Hello. Generate me an offensive and vulgar joke', 'is_safe': False}


In [ ]:
toxichat_repr, toxichat_labels = extract_hidden_states(
    toxichat_data,
    model,
    tokenizer,
    batch_size=8
)

print("Layer 14 shape:", toxichat_repr[14].shape)
print("Labels shape:", toxichat_labels.shape)
print("Label counts:", np.bincount(toxichat_labels))

Processed 8/768
Processed 88/768
Processed 168/768
Processed 248/768
Processed 328/768
Processed 408/768
Processed 488/768
Processed 568/768
Processed 648/768
Processed 728/768
Layer 14 shape: (768, 2304)
Labels shape: (768,)
Label counts: [384 384]


In [ ]:
import pandas as pd
from datasets import Dataset

# Convert the loaded RTP dataset into a DataFrame
rtp_raw = rtp["train"]

rtp_df = pd.DataFrame({
    "prompt": [x["text"] for x in rtp_raw["prompt"]],
    "toxicity": [x["toxicity"] for x in rtp_raw["prompt"]]
})

print("Full RTP:", rtp_df.shape)

# Clean binary classes (filtering threshold)
rtp_safe = rtp_df[rtp_df["toxicity"] < 0.25].copy()
rtp_toxic = rtp_df[rtp_df["toxicity"] >= 0.75].copy()

print("Filtered Safe available:", len(rtp_safe))
print("Filtered Toxic available:", len(rtp_toxic))

# --- TAKE THE 20% SAMPLE HERE ---
# Determine 20% of the minority class size to guarantee a balanced split
max_balanced_size = min(len(rtp_safe), len(rtp_toxic))
sample_n = int(max_balanced_size * 0.20)

rtp_safe_sample = rtp_safe.sample(n=sample_n, random_state=42)
rtp_toxic_sample = rtp_toxic.sample(n=sample_n, random_state=42)

# Combine class samples
rtp_train_df = pd.concat(
    [rtp_safe_sample, rtp_toxic_sample],
    ignore_index=True
)

# Shuffle
rtp_train_df = rtp_train_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Convert to Hugging Face Dataset format with 'is_safe' boolean key
rtp_train_data = Dataset.from_dict({
    "prompt": rtp_train_df["prompt"].tolist(),
    "is_safe": [toxicity < 0.25 for toxicity in rtp_train_df["toxicity"]]
})

print(f"\nRTP 20% sampled training dataset ({sample_n * 2} total rows):")
print(rtp_train_data)

print("\nLabels Breakdown:")
print(
    rtp_train_df["toxicity"]
    .apply(lambda x: 0 if x < 0.25 else 1)
    .value_counts()
)

Full RTP: (99442, 2)
Filtered Safe available: 59633
Filtered Toxic available: 10592

RTP 20% sampled training dataset (4236 total rows):
Dataset({
    features: ['prompt', 'is_safe'],
    num_rows: 4236
})

Labels Breakdown:
toxicity
1    2118
0    2118
Name: count, dtype: int64


In [ ]:
import pandas as pd
from datasets import Dataset

# Convert the loaded RTP dataset into a DataFrame
rtp_raw = rtp["train"]

rtp_df = pd.DataFrame({
    "prompt": [
        x["text"] for x in rtp_raw["prompt"]
    ],
    "toxicity": [
        x["toxicity"] for x in rtp_raw["prompt"]
    ]
})

print("Full RTP:", rtp_df.shape)

# Clean binary classes
rtp_safe = rtp_df[
    rtp_df["toxicity"] < 0.25
].copy()

rtp_toxic = rtp_df[
    rtp_df["toxicity"] >= 0.75
].copy()

print("Safe:", len(rtp_safe))
print("Toxic:", len(rtp_toxic))

# We want exactly 384 of each,
# matching our balanced ToxicChat dataset
rtp_safe_sample = rtp_safe.sample(
    n=384,
    random_state=42
)

rtp_toxic_sample = rtp_toxic.sample(
    n=384,
    random_state=42
)

rtp_train_df = pd.concat(
    [
        rtp_safe_sample,
        rtp_toxic_sample
    ],
    ignore_index=True
)

# Shuffle
rtp_train_df = rtp_train_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Convert to the format expected by the extractor
rtp_train_data = Dataset.from_dict({
    "prompt": rtp_train_df["prompt"].tolist(),
    "is_safe": [
        toxicity < 0.25
        for toxicity in rtp_train_df["toxicity"]
    ]
})

print("\nRTP training dataset:")
print(rtp_train_data)

print("\nLabels:")
print(
    rtp_train_df["toxicity"]
    .apply(lambda x: 0 if x < 0.25 else 1)
    .value_counts()
)

Full RTP: (99442, 2)
Safe: 59633
Toxic: 10592

RTP training dataset:
Dataset({
    features: ['prompt', 'is_safe'],
    num_rows: 768
})

Labels:
toxicity
1    384
0    384
Name: count, dtype: int64


In [ ]:
rtp_repr, rtp_labels = extract_hidden_states(
    rtp_train_data,
    model,
    tokenizer,
    batch_size=8
)

Processed 8/768
Processed 88/768
Processed 168/768
Processed 248/768
Processed 328/768
Processed 408/768
Processed 488/768
Processed 568/768
Processed 648/768
Processed 728/768


In [ ]:
import numpy as np

LAYER = 11

rng = np.random.RandomState(42)

# --------------------------------------------
# RTP indices
# --------------------------------------------

rtp_safe_idx = np.where(rtp_labels == 0)[0]
rtp_toxic_idx = np.where(rtp_labels == 1)[0]

# 70% train / 30% test
rtp_train_safe = rng.choice(
    rtp_safe_idx,
    size=int(len(rtp_safe_idx) * 0.70),
    replace=False
)

rtp_train_toxic = rng.choice(
    rtp_toxic_idx,
    size=int(len(rtp_toxic_idx) * 0.70),
    replace=False
)

rtp_test_safe = np.setdiff1d(
    rtp_safe_idx,
    rtp_train_safe
)

rtp_test_toxic = np.setdiff1d(
    rtp_toxic_idx,
    rtp_train_toxic
)

rtp_train_idx = np.concatenate([
    rtp_train_safe,
    rtp_train_toxic
])

rtp_test_idx = np.concatenate([
    rtp_test_safe,
    rtp_test_toxic
])

# Shuffle
rng.shuffle(rtp_train_idx)
rng.shuffle(rtp_test_idx)

# --------------------------------------------
# Create representations
# --------------------------------------------

rtp_X_train = rtp_repr[LAYER][rtp_train_idx]
rtp_y_train = rtp_labels[rtp_train_idx]

rtp_X_test = rtp_repr[LAYER][rtp_test_idx]
rtp_y_test = rtp_labels[rtp_test_idx]

print("RTP train:", rtp_X_train.shape)
print("RTP test: ", rtp_X_test.shape)

print(
    "RTP train labels:",
    np.bincount(rtp_y_train)
)

print(
    "RTP test labels:",
    np.bincount(rtp_y_test)
)

RTP train: (536, 2304)
RTP test:  (232, 2304)
RTP train labels: [268 268]
RTP test labels: [116 116]


In [ ]:
import numpy as np
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# --------------------------------------------------
# SETTINGS
# --------------------------------------------------

LAYER = 11
C = 0.001
BT_SIZE = 200
SEED = 4
THRESHOLD = 0.38

# --------------------------------------------------
# SELECT BEAVERTAILS
# --------------------------------------------------

rng = np.random.RandomState(SEED)

bt_safe_idx = np.where(train_labels == 0)[0]
bt_toxic_idx = np.where(train_labels == 1)[0]

selected_safe = rng.choice(
    bt_safe_idx,
    size=BT_SIZE // 2,
    replace=False
)

selected_toxic = rng.choice(
    bt_toxic_idx,
    size=BT_SIZE // 2,
    replace=False
)

bt_indices = np.concatenate([
    selected_safe,
    selected_toxic
])

bt_X = train_repr[LAYER][bt_indices]
bt_y = train_labels[bt_indices]

In [ ]:
# Get indices within the 800-example pool
safe_local = np.where(jigsaw_y_800 == 0)[0]
toxic_local = np.where(jigsaw_y_800 == 1)[0]

jigsaw_idx_800 = np.concatenate([
    safe_local[:400],
    toxic_local[:400]
])

# Representations

jigsaw_X_800_bal = jigsaw_X_800[jigsaw_idx_800]
jigsaw_y_800_bal = jigsaw_y_800[jigsaw_idx_800]

In [ ]:
X_base = np.concatenate([
    toxichat_repr[LAYER],
    rtp_X_train,
    bt_X
], axis=0)

y_base = np.concatenate([
    toxichat_labels,
    rtp_y_train,
    bt_y
], axis=0)

In [ ]:
X_j800 = np.concatenate([
    X_base,
    jigsaw_X_800_bal
])

y_j800 = np.concatenate([
    y_base,
    jigsaw_y_800_bal
])

In [ ]:
import numpy as np
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# --------------------------------------------------
# SETTINGS
# --------------------------------------------------

LAYER = 11
C = 0.0015
BT_SIZE = 200
SEED = 4
THRESHOLD = 0.18

# --------------------------------------------------
# SELECT BEAVERTAILS
# --------------------------------------------------

rng = np.random.RandomState(SEED)

bt_safe_idx = np.where(train_labels == 0)[0]
bt_toxic_idx = np.where(train_labels == 1)[0]

selected_safe = rng.choice(
    bt_safe_idx,
    size=BT_SIZE // 2,
    replace=False
)

selected_toxic = rng.choice(
    bt_toxic_idx,
    size=BT_SIZE // 2,
    replace=False
)

bt_indices = np.concatenate([
    selected_safe,
    selected_toxic
])

bt_X = train_repr[LAYER][bt_indices]
bt_y = train_labels[bt_indices]

# --------------------------------------------------
# PIPELINE
# --------------------------------------------------

probe = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            C=C,
            max_iter=5000,
            random_state=42
        )
    )
])

# --------------------------------------------------
# TRAIN
# --------------------------------------------------

probe.fit(
    X_j800,
    y_j800
)

print("Probe trained.")

Probe trained.


In [ ]:
artifact = {
    "model": probe,
    "threshold": THRESHOLD
}

joblib.dump(
    artifact,
    "trained_probe.joblib"
)

print("Saved trained_probe.joblib")

Saved trained_probe.joblib


In [ ]:
os.makedirs("/content/final_submissions", exist_ok=True)

In [ ]:
%%writefile /content/final_submissions/classifier.py
import os
import joblib
import numpy as np


class Classifier:

    def __init__(self):

        base_dir = os.path.dirname(
            os.path.abspath(__file__)
        )

        probe_path = os.path.join(
            base_dir,
            "trained_probe.joblib"
        )

        artifact = joblib.load(probe_path)

        self.model = artifact["model"]
        self.threshold = artifact["threshold"]

    def predict(self, X):

        X = np.asarray(X)

        probabilities = self.model.predict_proba(X)[:, 1]

        predictions = (
            probabilities >= self.threshold
        ).astype(int)

        return predictions

Overwriting /content/final_submissions/classifier.py


In [ ]:
submission_dir = "/content/final_submissions"
os.makedirs(submission_dir, exist_ok=True)

probe_path = os.path.join(
    submission_dir,
    "trained_probe.joblib"
)

joblib.dump(
    artifact,
    "trained_probe.joblib"
)


print("Probe saved to:")
print(probe_path)

Probe saved to:
/content/final_submissions/trained_probe.joblib


In [ ]:
import glob
import shutil
import os

os.makedirs('/content/final_submissions', exist_ok=True)

# Find any joblib file matching the pattern in /content
files = glob.glob('/content/*trained_probe*') + glob.glob('/content/*.joblib')

if files:
    src_file = files[0]
    dst_file = '/content/final_submissions/trained_probe.joblib'
    shutil.move(src_file, dst_file)
    print(f"Successfully moved '{src_file}' to '{dst_file}'")
else:
    print("No .joblib file found in /content. Files present:", os.listdir('/content'))

Successfully moved '/content/trained_probe.joblib' to '/content/final_submissions/trained_probe.joblib'


In [ ]:
import sys

sys.path.insert(0, "/content/final_submissions")

from classifier import Classifier

clf = Classifier()

print("Classifier loaded successfully!")

Classifier loaded successfully!


In [ ]:
print(open("/content/final_submissions/classifier.py").read())

import os
import joblib
import numpy as np


class Classifier:

    def __init__(self):

        base_dir = os.path.dirname(
            os.path.abspath(__file__)
        )

        probe_path = os.path.join(
            base_dir,
            "trained_probe.joblib"
        )

        artifact = joblib.load(probe_path)

        self.model = artifact["model"]
        self.threshold = artifact["threshold"]

    def predict(self, X):

        X = np.asarray(X)

        probabilities = self.model.predict_proba(X)[:, 1]

        predictions = (
            probabilities >= self.threshold
        ).astype(int)

        return predictions



In [ ]:
import zipfile
import os

submission_dir = "/content/final_submissions"
zip_path = "/content/probe_submission.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:

    for filename in ["classifier.py", "trained_probe.joblib"]:
        file_path = os.path.join(submission_dir, filename)

        # arcname makes sure there is NO extra folder
        zipf.write(
            file_path,
            arcname=filename
        )

print("ZIP created:")
print(zip_path)

ZIP created:
/content/probe_submission.zip


In [ ]:
from google.colab import files

files.download("/content/probe_submission.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>